In [33]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from scipy.spatial.distance import pdist

In [34]:
training_configs = np.load("LJ13_configs.npy")

In [35]:
com = training_configs.mean(axis=1, keepdims=True)

training_configs_centered = training_configs - com

In [36]:
training_configs_centered_tensor = torch.tensor(training_configs_centered, dtype=torch.float32)

print(training_configs_centered_tensor.shape)

torch.Size([24994, 13, 3])


In [37]:
class MessageMLP(nn.Module):

    def __init__(self, hidden_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2*hidden_dim + 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU()
        )

    def forward(self, x):
        return self.net(x)

In [38]:
class FeatureMLP(nn.Module):

    def __init__(self, hidden_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2*hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, x):
        return self.net(x)

In [39]:
class CoordMLP(nn.Module):

    def __init__(self, hidden_dim=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2*hidden_dim + 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1)
        )

        torch.nn.init.xavier_uniform_(
            self.net[-1].weight,
            gain=0.001
        )

        torch.nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(self, x):
        return self.net(x)

In [40]:
class EGNNLayer(nn.Module):

    def __init__(self, hidden_dim=64):
        super().__init__()

        self.message_mlp = MessageMLP(hidden_dim)

        self.feature_mlp = FeatureMLP(hidden_dim)

        self.coord_mlp = CoordMLP(hidden_dim)

    def forward(self, x, h, log_sigma):

        N = x.shape[1]

        diff = x[:,:,None,:] - x[:,None,:,:]
        dist2 = (diff**2).sum(dim=-1, keepdim=True)

        hi = h[:,:,None,:]
        hi = hi.expand(-1,-1,N,-1)

        hj = h[:,None,:,:]
        hj = hj.expand(-1,N,-1,-1)

        log_sigma_pair = log_sigma[:,None,None,:]
        log_sigma_pair = log_sigma_pair.expand(-1,N,N,-1)

        message_input = torch.cat([hi, hj, dist2, log_sigma_pair], dim=-1)

        messages = self.message_mlp(message_input)

        agg_messages = messages.sum(dim=2)

        feature_input = torch.cat([h, agg_messages], dim=-1)

        feature_update = self.feature_mlp(feature_input)

        h_new = h + feature_update

        coord_weights = self.coord_mlp(message_input)

        coord_update = diff * coord_weights

        delta_x = coord_update.sum(dim=2)

        delta_x = delta_x / x.shape[1]

        x_new = x + delta_x

        return x_new, h_new, delta_x

    

In [41]:
class ScoreNetwork(nn.Module):

    def __init__(
        self,
        hidden_dim=64,
        num_layers=3
    ):
        super().__init__()

        self.embedding = nn.Linear(1, hidden_dim)

        self.layers = nn.ModuleList(
            [
                EGNNLayer(hidden_dim)
                for _ in range(num_layers)
            ]
        )


    def forward(self, x, log_sigma):

        atom_features = torch.ones(x.shape[0], x.shape[1], 1)

        h = self.embedding(atom_features)

        f_pred = torch.zeros_like(x)

        for layer in self.layers:

            x, h, delta_x = layer(x, h, log_sigma)

            f_pred = f_pred + delta_x

        f_pred = f_pred - f_pred.mean(dim=1, keepdim=True)


        return f_pred


In [42]:
X = training_configs_centered.reshape(training_configs_centered.shape[0],39)
X = torch.tensor(X, dtype=torch.float32)

X_np = X.cpu().numpy()

sigma_max = pdist(X_np).max()
sigma_min = 0.01

print("sigma_max:", sigma_max)

sigma_max: 6.440439591617135


In [43]:
def get_sigma(t):

    return sigma_min * (sigma_max/sigma_min)**t

In [44]:
def train_model(
  optimizer_name="Adam",
  lr = 1e-4,
  batch_size = 100,
  width = 64,
  depth = 3,
  num_epochs = 600   
):
    dataset = TensorDataset(training_configs_centered_tensor)

    loader = DataLoader(dataset,batch_size=batch_size,shuffle=True)

    model = ScoreNetwork(hidden_dim=width,num_layers=depth)

    if optimizer_name == "Adam":
        
        optimizer=optim.Adam(
            model.parameters(),
            lr=lr
        )
    else:

        optimizer = optim.SGD(
            model.parameters(),
            lr=lr
        )
    

    loss_history = []

    for epoch in range(num_epochs):
        
        epoch_loss = 0.0

        for (x_batch,) in loader:
            
            batch_size_current = x_batch.shape[0]
            
            t = torch.rand(batch_size_current,1)

            sigma = get_sigma(t)

            sigma_expanded = sigma.view(-1,1,1)

            z = torch.randn_like(x_batch)

            z = z - z.mean(dim=1, keepdim=True)
            
            x_noisy = x_batch + sigma_expanded * z

            log_sigma = torch.log(sigma)
            
            target = -z  

            f_pred = model(x_noisy, log_sigma)

            loss_per_sample = ((f_pred - target) ** 2).sum(dim=(1,2))

            #weights = sigma.squeeze() ** 2

            loss = (loss_per_sample).mean()

            optimizer.zero_grad()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        loss_history.append(avg_loss)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:4d} | Loss = {avg_loss:.6f}")

    return model, loss_history

In [14]:
model, loss_history = train_model()

Epoch    0 | Loss = 23.783605
Epoch   10 | Loss = 13.839776
Epoch   20 | Loss = 11.859577
Epoch   30 | Loss = 12.017197
Epoch   40 | Loss = 11.933888
Epoch   50 | Loss = 11.224553
Epoch   60 | Loss = 11.182217
Epoch   70 | Loss = 11.144262
Epoch   80 | Loss = 10.845816
Epoch   90 | Loss = 10.686170
Epoch  100 | Loss = 10.663939
Epoch  110 | Loss = 10.533498
Epoch  120 | Loss = 10.602551
Epoch  130 | Loss = 10.538854
Epoch  140 | Loss = 10.508999
Epoch  150 | Loss = 10.357957
Epoch  160 | Loss = 10.405449
Epoch  170 | Loss = 10.255960
Epoch  180 | Loss = 10.444464
Epoch  190 | Loss = 10.343095
Epoch  200 | Loss = 10.342046
Epoch  210 | Loss = 10.396323
Epoch  220 | Loss = 10.307908
Epoch  230 | Loss = 10.181184
Epoch  240 | Loss = 10.249450
Epoch  250 | Loss = 10.187300
Epoch  260 | Loss = 10.218002
Epoch  270 | Loss = 10.277536
Epoch  280 | Loss = 10.241470
Epoch  290 | Loss = 10.178445
Epoch  300 | Loss = 10.287551
Epoch  310 | Loss = 10.154851
Epoch  320 | Loss = 10.157790
Epoch  330

In [13]:
torch.save(model.state_dict(), "lj13_f_theta_strategy_1.pth")

NameError: name 'model' is not defined

In [45]:
model = ScoreNetwork(hidden_dim=64, num_layers=3)

model.load_state_dict(torch.load("lj13_f_theta_2.pth"))

model.eval()

ScoreNetwork(
  (embedding): Linear(in_features=1, out_features=64, bias=True)
  (layers): ModuleList(
    (0-2): 3 x EGNNLayer(
      (message_mlp): MessageMLP(
        (net): Sequential(
          (0): Linear(in_features=130, out_features=64, bias=True)
          (1): SiLU()
          (2): Linear(in_features=64, out_features=64, bias=True)
          (3): SiLU()
        )
      )
      (feature_mlp): FeatureMLP(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=64, bias=True)
          (1): SiLU()
          (2): Linear(in_features=64, out_features=64, bias=True)
        )
      )
      (coord_mlp): CoordMLP(
        (net): Sequential(
          (0): Linear(in_features=130, out_features=64, bias=True)
          (1): SiLU()
          (2): Linear(in_features=64, out_features=64, bias=True)
          (3): SiLU()
          (4): Linear(in_features=64, out_features=1, bias=True)
        )
      )
    )
  )
)

In [46]:
def lj_energy(coords):

    N = coords.shape[0]

    energy = 0.0

    for i in range(N):
        for j in range(i + 1, N):

            r = torch.norm(coords[i] - coords[j])

            inv_r6 = (1.0 / r) ** 6
            inv_r12 = inv_r6 ** 2

            energy += 4 * (inv_r12 - inv_r6)

    return energy

In [47]:
def lj_energy_batch(coords):
    # coords shape: [batch, N, 3]

    diff = coords[:, :, None, :] - coords[:, None, :, :]

    # Pairwise distances
    r = torch.norm(diff, dim=-1)

    # Only take i < j
    i, j = torch.triu_indices(
        coords.shape[1],
        coords.shape[1],
        offset=1,
        device=coords.device
    )

    r_pairs = r[:, i, j]

    inv_r6 = (1.0 / r_pairs) ** 6
    inv_r12 = inv_r6 ** 2

    energy = 4 * (inv_r12 - inv_r6)

    # Sum over all 78 pairs
    return energy.sum(dim=1)

In [48]:
import torch.nn.functional as F

In [49]:
n_configs = 100
n_noise = 100
Temp = 0.1
beta = 1.0 / Temp
x0 = training_configs_centered_tensor[:n_configs]

sigma_values = [0.1, 0.05, 0.02, 0.01]

In [52]:
results = []

for sigma_value in sigma_values:

    sigma = torch.tensor(sigma_value)

    cos_noise_all = []
    cos_physics_all = []
    norm_ratio_noise_all = []
    norm_ratio_physics_all = []

    for repeat in range(n_noise):


        z = torch.randn_like(x0)
        z = z - z.mean(dim=1, keepdim=True)


        x_tilde = x0 + sigma * z


        log_sigma = torch.full((n_configs, 1), np.log(sigma_value))

        with torch.no_grad():
            f_pred = model(x_tilde, log_sigma)

        score_model = f_pred / sigma
        score_noise = -z / sigma


        x_phys = (x_tilde.detach().clone().requires_grad_(True))

        energies = lj_energy_batch(x_phys)
        total_energy = energies.sum()

        grad_U = torch.autograd.grad(total_energy, x_phys)[0]

        score_physics = -beta * grad_U


        model_flat = score_model.reshape(n_configs, -1)
        noise_flat = score_noise.reshape(n_configs, -1)
        physics_flat = score_physics.reshape(n_configs, -1)

        cos_noise = F.cosine_similarity(model_flat, noise_flat, dim=1)

        cos_physics = F.cosine_similarity(model_flat, physics_flat, dim=1)

        model_norm = torch.norm(model_flat, dim=1)
        noise_norm = torch.norm(noise_flat, dim=1)
        physics_norm = torch.norm(physics_flat, dim=1)

        norm_ratio_noise = model_norm / noise_norm
        norm_ratio_physics = model_norm / physics_norm

        cos_noise_all.extend(cos_noise.detach().cpu().numpy())

        cos_physics_all.extend(cos_physics.detach().cpu().numpy())

        norm_ratio_noise_all.extend(norm_ratio_noise.detach().cpu().numpy())
        norm_ratio_physics_all.extend(norm_ratio_physics.detach().cpu().numpy())


    results.append({
        "sigma": sigma_value,
        "cos_noise": np.mean(cos_noise_all),
        "cos_physics": np.mean(cos_physics_all),
        "norm_ratio_noise": np.mean(norm_ratio_noise_all),
        "norm_ratio_physics": np.mean(norm_ratio_physics_all)
    })

In [53]:
for result in results:
    print(
        f"sigma = {result['sigma']:.4f}, "
        f"cos_noise = {result['cos_noise']:.4f}, "
        f"cos_physics = {result['cos_physics']:.4f}, "
        f"norm_noise = {result['norm_ratio_noise']:.4f}, "
        f"norm_physics = {result['norm_ratio_physics']:.4f}"
    )

sigma = 0.1000, cos_noise = 0.8973, cos_physics = 0.5384, norm_noise = 0.9024, norm_physics = 0.0113
sigma = 0.0500, cos_noise = 0.7845, cos_physics = 0.7372, norm_noise = 0.8106, norm_physics = 0.0988
sigma = 0.0200, cos_noise = 0.5194, cos_physics = 0.9312, norm_noise = 0.5644, norm_physics = 0.4778
sigma = 0.0100, cos_noise = 0.3147, cos_physics = 0.9711, norm_noise = 0.3501, norm_physics = 0.7936
